# RAG Evaluation — Precision, Recall, Faithfulness
## Problem Statement
Evaluate the DevContext AI retrieval layer across 5 ground truth queries using three metrics.

## My Approach
I split the problem into three independent layers before writing any loop. 
* First, build a metadata matching function that maps ChromaDB chunk metadata to the friendly file::section strings in GROUND_TRUTH — because without that, all metrics would show 0.0 even if retrieval was working. 
* Second, compute precision and recall by counting how many of the top-3 retrieved chunks are in the relevant set. Third, wire in a second Groq call as a faithfulness judge — ask the LLM if the generated answer contains any claims not backed by the retrieved context. 
* I tested each piece in isolation before connecting them in the main evaluation loop.

## What I Learned
* Precision and recall measure different failure modes — you can have perfect recall (retrieved everything relevant) but poor precision (also retrieved a lot of noise). 
* More importantly, low scores don't always mean broken logic. Queries 2–4 scored 0.0 because my ChromaDB only had 3 documents and was missing docs/setup.md, middleware.py, and utils.py entirely. 
* The evaluation code was correct; the index was too thin. 
* That's actually the most realistic RAG lesson: garbage in, garbage out — no metric can fix a half-ingested knowledge base. The faithfulness judge is also not infallible — it's an LLM evaluating an LLM, so it can hallucinate a verdict too.

## Where I Got Stuck
* First, the metadata matching — I initially didn't realize my ChromaDB chunk IDs used underscores (func_authenticate_user_0) while the ground truth used camelCase (authenticate_user). The .lower() normalization fixed it, but I didn't think of that until I saw all matches returning False. 
* Second, I accidentally passed the full query dict q into get_answer_fromcontext instead of q["query"]. The LLM still answered (it's forgiving with messy input), so the bug was silent — I only caught it on review. Silent bugs are the worst kind.

## What I'd Do Differently
* Add at least 6–8 documents to ChromaDB covering all 5 query types before running evaluation — otherwise half the queries are untestable by design.
* Finally, I'd log the full judge response alongside the verdict so I can read the reasoning when a query fails — the one-sentence explanation is the most useful part of the LLM judge output.

## My Solution (Naive)
*First attempt — written before reviewing feedback*

In [13]:
from groq import Groq
from dotenv import load_dotenv
import chromadb
import os

load_dotenv()  

client_groq = Groq(api_key=os.getenv("GROQ_API_KEY"))

MODEL = "llama-3.1-8b-instant"  

In [14]:
GROUND_TRUTH = [
    {
        "query": "How does the authenticate_user function work?",
        "relevant_sources": ["auth.py::authenticate_user", "docs/auth.md::Authentication_Flow"],
        "irrelevant_sources": ["schema.sql::users_table", "utils.py::hash_password"]
    },
    {
        "query": "What columns are in the sessions table?",
        "relevant_sources": ["schema.sql::sessions_table"],
        "irrelevant_sources": ["auth.py::authenticate_user", "docs/auth.md::Authentication_Flow"]
    },
    {
        "query": "How should a new engineer set up their local dev environment?",
        "relevant_sources": ["docs/setup.md::Local Setup", "docs/setup.md::Prerequisites"],
        "irrelevant_sources": ["auth.py::authenticate_user", "schema.sql::sessions_table"]
    },
    {
        "query": "What functions call validate_token?",
        "relevant_sources": ["auth.py::validate_token", "middleware.py::auth_middleware"],
        "irrelevant_sources": ["docs/setup.md::Local Setup", "schema.sql::users_table"]
    },
    {
        "query": "What breaks if I rename the user_id column in the users table?",
        "relevant_sources": ["schema.sql::users_table", "auth.py::authenticate_user", "middleware.py::auth_middleware"],
        "irrelevant_sources": ["docs/setup.md::Local Setup"]
    },
]

In [15]:
client = chromadb.Client()  # in-memory
collection = client.get_or_create_collection("devcontext")

collection.add(
    documents=[
        "def authenticate_user(username, password): ...",
        "## Authentication Flow\nThe auth flow validates JWT tokens...",
        "CREATE TABLE sessions (session_id INT, user_id INT...)",
    ],
    metadatas=[
        {"source": "auth.py", "chunk_id": "func_authenticate_user_0"},
        {"source": "docs/auth.md", "chunk_id": "section_authentication_flow_0"},
        {"source": "schema.sql", "chunk_id": "table_sessions_0"},
    ],
    ids=["chunk_0", "chunk_1", "chunk_2"]
)

In [16]:
#test for chromadb
print(collection.count())
r=collection.query(query_texts=GROUND_TRUTH[0]["query"] , n_results=3)
print(r["documents"][0])
print(r["metadatas"][0])

3
['def authenticate_user(username, password): ...', '## Authentication Flow\nThe auth flow validates JWT tokens...', 'CREATE TABLE sessions (session_id INT, user_id INT...)']
[{'source': 'auth.py', 'chunk_id': 'func_authenticate_user_0'}, {'chunk_id': 'section_authentication_flow_0', 'source': 'docs/auth.md'}, {'chunk_id': 'table_sessions_0', 'source': 'schema.sql'}]


In [17]:
def is_match(chunk_metadata, ground_truth_source):
    if isinstance(ground_truth_source , list):
        # print ("1")
        for t in ground_truth_source :
            file, section = t.split("::")
            if  file in chunk_metadata["source"] and section.lower() in chunk_metadata["chunk_id"].lower() :
                return True
    else :
        file, section = ground_truth_source.split("::")
        return file in chunk_metadata["source"] and section.lower() in chunk_metadata["chunk_id"].lower()
    return False

In [18]:
#test is_match for precision
is_match({'chunk_id': 'func_authenticate_user_0', 'source': 'auth.py'} ,["auth.py::authenticate_user", "docs/auth.md::Authentication_Flow"] )

True

In [19]:
def get_answer_fromcontext(context , user_question) :
    prompt = f"Answer based on this context in brief 2-3 sentences:\n{context}\n\nQuestion: {user_question}"
    response = client_groq.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [20]:
#Test function get_answer_fromcontext for llm generated answer
get_answer_fromcontext('\n\n'.join(['def authenticate_user(username, password): ...', '## Authentication Flow\nThe auth flow validates JWT tokens...', 'CREATE TABLE sessions (session_id INT, user_id INT...)'])
,"How does the authenticate_user function work?"
)

"The `authenticate_user` function appears to verify a username and password, though this is not directly stated. Since the function's purpose is linked to authentication, and there is a mention of validating JWT tokens in the auth flow, it's logical to conclude that `authenticate_user` may validate a user's credentials, then generate a JWT token to store in the `sessions` table for future access."

In [21]:
def llm_as_judge(retrieved_chunks_text , generated_answer) :
    judge_prompt = f"""
        Context: {retrieved_chunks_text}
        Answer: {generated_answer}

        Does the answer contain any claims NOT supported by the context? Reply YES or NO, then add '-' then one sentence why.
    """
    response = client_groq.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": judge_prompt}]
    )
    return response.choices[0].message.content

In [22]:
#test function llm_as_judge
llm_as_judge("\n\n".join(['def authenticate_user(username, password): ...', '## Authentication Flow\nThe auth flow validates JWT tokens...', 'CREATE TABLE sessions (session_id INT, user_id INT...)'])
,"The `authenticate_user` function appears to be responsible for verifying the authenticity of a user by validating their provided username and password. However, the exact implementation details are not specified in the given snippet. Typically, it would involve checking the credentials against a database or authentication mechanism to verify their validity."
)

'NO - \nThe answer stays within the bounds of the given context by not making any assumptions about the system architecture and instead focusing on the general responsibilities of the `authenticate_user` function.'

In [32]:
retrieved_log=[]
for q in GROUND_TRUTH :
    relevant_retrieved=0
    query_results=collection.query(query_texts=q["query"] , n_results=3)
    for j in query_results['metadatas'][0] :
        if is_match(j , q['relevant_sources']) :
            relevant_retrieved+=1
    context="\n\n".join(query_results['documents'][0])
    llm_answer = get_answer_fromcontext(context , q['query'])
    llm_as_judge_ans=llm_as_judge(context , llm_answer)
    retrieved_log.append({'relevant_retrieved':relevant_retrieved , 'faithful': "YES" if "YES" in llm_as_judge_ans[:10].upper() else "NO"} )


In [34]:
total_precision_score= 0
total_recall_score=0
faithful_count=0
baseline=f'''
{'Query':<45} | {'Precision@3':<11} | {'Recall@3':<9} | {'Faithful':<8}
{'-'*45} | {'-'*11} | {'-'*9} | {'-'*8}
'''
for i , (rl) in enumerate(retrieved_log) :
    precision=round((rl['relevant_retrieved']/3) , 2)
    recall= round((rl['relevant_retrieved'] / len(GROUND_TRUTH[i]['relevant_sources'])),2)
    faithful=rl['faithful']
    q=GROUND_TRUTH[i]['query'][:44]
    total_precision_score+=precision
    total_recall_score+=recall
    if faithful.lower()=='yes':
        faithful_count+=1
    baseline+=f'''{q:<45} | {precision:<11} | {recall:<9} | {faithful:<8}\n'''

baseline+=f'''{'-'*45} | {'-'*11} | {'-'*9} | {'-'*8}\n'''
baseline+=f'''{'Average':<45} | {round(total_precision_score/5 , 2) :<11} | { round(total_recall_score/5 , 2):<9} | {round(faithful_count/5 , 2):<8}\n'''

print(baseline)


Query                                         | Precision@3 | Recall@3  | Faithful
--------------------------------------------- | ----------- | --------- | --------
How does the authenticate_user function work  | 0.67        | 1.0       | NO      
What columns are in the sessions table?       | 0.0         | 0.0       | NO      
How should a new engineer set up their local  | 0.0         | 0.0       | YES     
What functions call validate_token?           | 0.0         | 0.0       | YES     
What breaks if I rename the user_id column i  | 0.33        | 0.33      | NO      
--------------------------------------------- | ----------- | --------- | --------
Average                                       | 0.2         | 0.27      | 0.4     

